# Chuyển đổi `star_count` và `forks_count` thành nhãn phân loại

Notebook này thêm hai cột `star-bin` và `fork-bin`, mỗi cột có ba nhãn: **thấp**, **trung bình**, **cao**.

## Phương pháp
Sử dụng phân loại theo **phân vị (quantile)**: chia các giá trị không thiếu của từng cột thành ba nhóm có số lượng bản ghi gần tương đương nhau. Cách này phù hợp khi dữ liệu GitHub có độ lệch lớn giữa các repository.

- `star_count` → `star-bin`
- `forks_count` → `fork-bin`

Giá trị thiếu sẽ được giữ là `NA` ở cột phân loại tương ứng. Hai cột số gốc không bị thay đổi. Notebook đọc `github_repos_complete.csv` và ghi kết quả ra file mới để không ghi đè dữ liệu nguồn.

In [31]:
from pathlib import Path
import pandas as pd

# File đầu vào cần nằm cùng thư mục với notebook; có thể sửa đường dẫn nếu cần.
INPUT_FILE = Path("github_repos_scale.csv")
OUTPUT_FILE = Path("github_repos_binned.csv")

STAR_COL = "star_count"
FORK_COL = "forks_count"
STAR_BIN_COL = "star-bin"
FORK_BIN_COL = "fork-bin"
LABELS = ["thấp", "trung bình", "cao"]

In [32]:
# Đọc dữ liệu và kiểm tra các cột cần thiết
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Không tìm thấy {INPUT_FILE.resolve()}. "
        "Hãy đặt file CSV cạnh notebook hoặc cập nhật INPUT_FILE."
    )

df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")
required_cols = [STAR_COL, FORK_COL]
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Thiếu cột bắt buộc trong CSV: {missing_cols}")

print(f"Số dòng dữ liệu: {len(df):,}")
print("Các cột được sử dụng:", required_cols)
display(df[required_cols].head())

Số dòng dữ liệu: 12,000
Các cột được sử dụng: ['star_count', 'forks_count']


,star_count,forks_count
0,11.253704,8.826564
1,6.401193,1.229446
2,4.102052,3.288174
3,2.824405,3.296514
4,2.662369,0.930407


## 1. Chuẩn hóa kiểu dữ liệu
Chuyển hai cột đầu vào sang dạng số. Giá trị không chuyển đổi được sẽ thành `NA`; các cột gốc được chuyển đổi kiểu số trong DataFrame để phân loại chính xác.

In [33]:
for col in [STAR_COL, FORK_COL]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

display(df[[STAR_COL, FORK_COL]].describe().T)

,count,mean,std,min,25%,50%,75%,max
star_count,12000.0,1.894781e-17,1.000042,-0.299346,-0.265829,-0.217884,-0.086923,28.737361
forks_count,12000.0,1.421085e-17,1.000042,-0.238747,-0.203800,-0.169646,-0.083072,42.551333


## 2. Tạo nhãn thấp / trung bình / cao
Dùng `qcut` với ba phân vị. `duplicates='drop'` giúp xử lý trường hợp dữ liệu có quá ít giá trị khác nhau; nếu không thể tạo đủ ba nhóm thì notebook sẽ báo để bạn kiểm tra thay vì gán nhãn sai.

In [34]:
def assign_quantile_bins(series: pd.Series, labels=LABELS) -> pd.Series:
    """Phân loại một cột thành thấp/trung bình/cao theo phân vị."""
    result = pd.Series(pd.NA, index=series.index, dtype="string")
    valid = series.dropna()

    if valid.empty:
        return result

    # qcut có thể trả về ít hơn 3 nhóm nếu có nhiều giá trị trùng lặp.
    binned = pd.qcut(valid, q=3, labels=False, duplicates="drop")
    number_of_bins = int(binned.nunique())
    if number_of_bins != 3:
        raise ValueError(
            f"Không thể chia cột thành đủ 3 nhóm theo phân vị (chỉ tạo được {number_of_bins} nhóm). "
            "Hãy cân nhắc dùng ngưỡng cố định hoặc kiểm tra các giá trị duy nhất."
        )

    result.loc[valid.index] = binned.map({0: labels[0], 1: labels[1], 2: labels[2]}).astype("string")
    return result

df[STAR_BIN_COL] = assign_quantile_bins(df[STAR_COL])
df[FORK_BIN_COL] = assign_quantile_bins(df[FORK_COL])

display(df[[STAR_COL, STAR_BIN_COL, FORK_COL, FORK_BIN_COL]].head(10))

,star_count,star-bin,forks_count,fork-bin
0,11.253704,cao,8.826564,cao
1,6.401193,cao,1.229446,cao
2,4.102052,cao,3.288174,cao
3,2.824405,cao,3.296514,cao
4,2.662369,cao,0.930407,cao
5,2.126835,cao,0.913330,cao
6,1.883854,cao,0.910947,cao
7,1.523897,cao,0.817224,cao
8,1.482019,cao,0.100006,cao
9,1.454125,cao,0.928024,cao


## 3. Kiểm tra kết quả
Bảng dưới cho biết số repository thuộc mỗi nhóm. Do phân loại theo phân vị, số lượng mỗi nhóm thường gần bằng nhau, nhưng có thể khác nhau khi nhiều repository có cùng số sao hoặc fork.

In [35]:
print("Phân bố star-bin:")
display(df[STAR_BIN_COL].value_counts(dropna=False).rename_axis(STAR_BIN_COL).to_frame("count"))

print("Phân bố fork-bin:")
display(df[FORK_BIN_COL].value_counts(dropna=False).rename_axis(FORK_BIN_COL).to_frame("count"))

print("Ngưỡng phân vị tham khảo:")
for col in [STAR_COL, FORK_COL]:
    print(f"\\n{col}")
    print(df[col].quantile([0, 1/3, 2/3, 1]).to_string())

Phân bố star-bin:


,count
star-bin,
cao,4000
trung bình,4000
thấp,4000


Phân bố fork-bin:


,count
fork-bin,
thấp,4015
cao,3996
trung bình,3989


Ngưỡng phân vị tham khảo:
\nstar_count
0.000000    -0.299346
0.333333    -0.253942
0.666667    -0.151787
1.000000    28.737361
\nforks_count
0.000000    -0.238747
0.333333    -0.194666
0.666667    -0.122785
1.000000    42.551333


## 4. Lưu dữ liệu
Lưu thành CSV mới, giữ nguyên các cột khác và bổ sung `star-bin`, `fork-bin`.

In [36]:
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"Đã lưu file: {OUTPUT_FILE.resolve()}")
print("Hai cột mới:", STAR_BIN_COL, "và", FORK_BIN_COL)

Đã lưu file: D:\HomeWork\IE313\Đồ án\analysis-popularity-github-repository\notebooks\chuyen_doi_phan_loai\github_repos_binned.csv
Hai cột mới: star-bin và fork-bin
